# XModalix3D three-modality alignment — compact diagnostic notebook

This notebook trains **XModalix3D** on paired NCCT, CBF and CBV volumes with **50 unimodal pretraining epochs per modality** followed by **100 joint XModalix3D epochs**.

The diagnostic output is deliberately restricted to the quantities that directly answer the current alignment question:

- **External modality probe**: can modality still be decoded from the latent space? Lower is better; chance is 1/3.
- **Patient-specific matched/unmatched distance ratio**: are the same patient's cross-modal embeddings closer than embeddings from different patients? Lower than 1 is better.
- **Internal adversarial classifier accuracy**: does the classifier used inside XModalix3D itself distinguish modalities? This is compared with the external probe, but is *not* used as proof of alignment.
- **Posterior sigma**: indicates whether `z` differs materially from `mu` or the posterior becomes nearly deterministic.
- **Alignment-related training losses**: adversarial, paired and classifier losses; final reconstruction/variational losses are retained as a compact training-health summary.

Checkpoint trajectories are evaluated on the **validation split**, because validation embeddings are captured with a fixed model after each training epoch. Train-batch checkpoint embeddings are not used for the trajectory diagnostics because they are collected while model parameters change within the epoch. Test metrics are shown only in the final frozen summary and should not be used for hyperparameter selection.


## Environment and configuration

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import autoencodix as acx

from autoencodix.configs import DataCase, DataConfig, DataInfo
from autoencodix.configs.xmodalix3d_config import XModalix3DConfig

DATA_ROOT = Path(
    os.environ.get(
        "DATA_ROOT",
        "/data/horse/ws/baeuchl-imagix3d/data/stroke_data",
    )
)

ncctpath = os.environ.get("NCCT_PATH", str(DATA_ROOT / "ncct_flat"))
ncctannopath = os.environ.get("NCCT_ANNO", str(DATA_ROOT / "ncct_anno_dst.csv"))
cbfpath = os.environ.get("CBF_PATH", str(DATA_ROOT / "cbf_flat"))
cbfannopath = os.environ.get("CBF_ANNO", str(DATA_ROOT / "cbf_anno_dst.csv"))
cbvpath = os.environ.get("CBV_PATH", str(DATA_ROOT / "cbv_flat"))
cbvannopath = os.environ.get("CBV_ANNO", str(DATA_ROOT / "cbv_anno_dst.csv"))

paths_to_check = {
    "NCCT volumes": Path(ncctpath),
    "NCCT annotation": Path(ncctannopath),
    "CBF volumes": Path(cbfpath),
    "CBF annotation": Path(cbfannopath),
    "CBV volumes": Path(cbvpath),
    "CBV annotation": Path(cbvannopath),
}
for name, path in paths_to_check.items():
    assert path.exists(), f"Missing input: {name}: {path}"

assert torch.cuda.is_available(), "This HPC notebook is configured for CUDA."

config = XModalix3DConfig(
    data_case=DataCase.IMG_TO_IMG,

    # 3D handling
    img_path_col="filepath",
    spatial_shape_policy="crop_or_pad_to_shape",
    target_shape_3d=(160, 192, 160),
    volume_scaling_strategy="per_volume",
    normalize_nonzero_only=True,

    # Joint XModalix3D training
    epochs=100,
    checkpoint_interval=5,
    batch_size=35,
    latent_dim=64,
    hidden_dim=16,

    # Pretraining
    pretrain_epochs=50,

    # Losses / optimization
    delta_class=0,
    reconstruction_loss="mse",
    loss_reduction="mean",
    scaling="NONE",
    learning_rate=0.00001,
    weight_decay=0.00001,
    beta=0.00001,
    anneal_function="logistic-late",
    gamma=30,
    delta_pair=5,

    # Reproducibility
    reproducible=True,
    global_seed=1,

    # 3D normalization / posterior handling
    train_normalization="group",
    train_norm_groupsize=8,
    clamp_logvar=True,
    keep_mu_positive=False,

    requires_paired=True,
    device="cuda",
    n_gpus=1,

    data_config=DataConfig(
        data_info={
            "NCCT": DataInfo(
                file_path=ncctpath,
                extra_anno_file=ncctannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction="from",
                pretrain_epochs=50,
            ),
            "CBF": DataInfo(
                file_path=cbfpath,
                extra_anno_file=cbfannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction="to",
                pretrain_epochs=50,
            ),
            "CBV": DataInfo(
                file_path=cbvpath,
                extra_anno_file=cbvannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction=None,
                pretrain_epochs=50,
            ),
            "ANNO": DataInfo(
                file_path=ncctannopath,
                data_type="ANNOTATION",
            ),
        }
    ),
)

print({
    "python": sys.executable,
    "autoencodix": acx.__file__,
    "gpu": torch.cuda.get_device_name(0),
    "joint_epochs": config.epochs,
    "pretrain_epochs": {
        key: value.pretrain_epochs
        for key, value in config.data_config.data_info.items()
        if value.data_type == "IMG"
    },
    "checkpoint_interval": config.checkpoint_interval,
    "latent_dim": config.latent_dim,
    "gamma": config.gamma,
    "delta_pair": config.delta_pair,
    "beta": config.beta,
})


## Memory-saving checkpoint capture

For alignment diagnostics we only need `mu`, sampled `z`, latent log-variance and sample IDs at checkpoints. Full 3D reconstructions are therefore omitted from checkpoint capture to avoid unnecessary RAM use. This patch changes only what is stored for diagnostics; it does not change the forward pass, losses, gradients or optimizer updates.


In [ ]:
from autoencodix.trainers._xmodal_trainer import XModalTrainer


def _capture_alignment_dynamics(self, batch_data):
    captured_data = {
        "latentspaces": {},
        "reconstructions": {},
        "mus": {},
        "sigmas": {},
        "sample_ids": {},
    }

    for mod_name, dynamics in self._modality_dynamics.items():
        sample_ids = batch_data[mod_name].get("sample_ids")
        if sample_ids is not None:
            captured_data["sample_ids"][mod_name] = np.asarray(sample_ids)

        model_output = dynamics["mp"]

        if model_output.latentspace is not None:
            captured_data["latentspaces"][mod_name] = model_output.latentspace.detach()
        if model_output.latent_mean is not None:
            captured_data["mus"][mod_name] = model_output.latent_mean.detach()
        if model_output.latent_logvar is not None:
            # AUTOENCODIX stores latent_logvar in result.sigmas.
            captured_data["sigmas"][mod_name] = model_output.latent_logvar.detach()

    return captured_data


XModalTrainer._capture_dynamics = _capture_alignment_dynamics
print("Alignment checkpoint capture enabled (mu, z, logvar, sample IDs only).")


## Preprocess and verify pairing

In [ ]:
xmodal3d = acx.XModalix3D(config=config)
xmodal3d.preprocess()

expected_modalities = {"img.NCCT", "img.CBF", "img.CBV"}
rows = []

for split_name in ["train", "valid", "test"]:
    split = getattr(xmodal3d._datasets, split_name)
    assert split is not None, f"Missing {split_name} split."
    assert set(split.datasets.keys()) == expected_modalities
    assert split.is_fully_paired

    id_sets = [set(ds.sample_ids) for ds in split.datasets.values()]
    assert all(ids == id_sets[0] for ids in id_sets[1:])

    for modality, ds in split.datasets.items():
        rows.append({
            "split": split_name,
            "modality": modality,
            "n": len(ds),
            "shape": tuple(ds[0][1].shape),
        })

pairing_check = pd.DataFrame(rows)
display(pairing_check)
print("Pairing checks passed.")


## Train XModalix3D

In [ ]:
xmodal3d.fit()

## Diagnostic helpers

The external probe is a fresh logistic-regression classifier trained only for diagnosis on frozen embeddings, using patient-grouped cross-validation. The internal classifier is the adversarial classifier that was trained jointly inside XModalix3D. Detailed confusion matrices and modality-pair reports are retained in variables but are not printed by default.


In [ ]:
from itertools import combinations
from torch.utils.data import DataLoader
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

from autoencodix.data._multimodal_dataset import create_multimodal_collate_fn


def extract_frozen_embeddings(trainer, dataset, batch_size):
    """Extract mu, z, logvar and sample IDs with the final model held fixed."""
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=create_multimodal_collate_fn(multimodal_dataset=dataset),
    )

    mu_parts = {mod: [] for mod in dataset.datasets.keys()}
    z_parts = {mod: [] for mod in dataset.datasets.keys()}
    logvar_parts = {mod: [] for mod in dataset.datasets.keys()}
    id_parts = {mod: [] for mod in dataset.datasets.keys()}

    for dynamics in trainer._modality_dynamics.values():
        dynamics["model"].eval()

    with torch.inference_mode():
        for batch in loader:
            for mod_name, mod_data in batch.items():
                model = trainer._modality_dynamics[mod_name]["model"]
                x = mod_data["data"].to(trainer._fabric.device)
                output = model(x)

                mu_parts[mod_name].append(output.latent_mean.detach().cpu().numpy())
                z_parts[mod_name].append(output.latentspace.detach().cpu().numpy())
                logvar_parts[mod_name].append(output.latent_logvar.detach().cpu().numpy())
                id_parts[mod_name].extend(mod_data["sample_ids"])

    return (
        {mod: np.concatenate(parts, axis=0) for mod, parts in mu_parts.items()},
        {mod: np.concatenate(parts, axis=0) for mod, parts in z_parts.items()},
        {mod: np.concatenate(parts, axis=0) for mod, parts in logvar_parts.items()},
        {mod: np.asarray(ids).astype(str) for mod, ids in id_parts.items()},
    )


def get_aligned_pair(latent_dict, id_dict, modality_a, modality_b):
    ids_a = np.asarray(id_dict[modality_a]).astype(str)
    ids_b = np.asarray(id_dict[modality_b]).astype(str)
    z_a = np.asarray(latent_dict[modality_a])
    z_b = np.asarray(latent_dict[modality_b])

    idx_a = {sid: i for i, sid in enumerate(ids_a)}
    idx_b = {sid: i for i, sid in enumerate(ids_b)}
    common_ids = sorted(set(ids_a) & set(ids_b))

    A = np.stack([z_a[idx_a[sid]] for sid in common_ids])
    B = np.stack([z_b[idx_b[sid]] for sid in common_ids])
    return common_ids, A, B


def pairwise_alignment_report_exact(latent_dict, id_dict):
    """Matched cross-modal distance divided by all unmatched-patient distances."""
    rows = []
    for a, b in combinations(sorted(latent_dict.keys()), 2):
        common_ids, A, B = get_aligned_pair(latent_dict, id_dict, a, b)

        matched_mean = np.linalg.norm(A - B, axis=1).mean()
        distance_matrix = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)
        unmatched_mean = distance_matrix[~np.eye(len(common_ids), dtype=bool)].mean()

        rows.append({
            "modality_a": a,
            "modality_b": b,
            "n_pairs": len(common_ids),
            "matched_distance": matched_mean,
            "unmatched_distance": unmatched_mean,
            "matched/unmatched_ratio": matched_mean / unmatched_mean,
        })
    return pd.DataFrame(rows)


def build_modality_probe_data(latent_dict, id_dict):
    X, y, groups = [], [], []
    for modality, latent in latent_dict.items():
        ids = np.asarray(id_dict[modality]).astype(str)
        for row, sid in zip(np.asarray(latent), ids):
            X.append(row)
            y.append(modality)
            groups.append(sid)
    return np.asarray(X), np.asarray(y), np.asarray(groups)


def modality_probe_score(latent_dict, id_dict):
    """Independent external modality probe with patient-grouped CV."""
    X, y, groups = build_modality_probe_data(latent_dict, id_dict)
    modalities = sorted(np.unique(y))
    n_groups = len(np.unique(groups))

    cv = GroupKFold(n_splits=min(5, n_groups))
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, class_weight="balanced"),
    )

    pred = cross_val_predict(
        probe,
        X,
        y,
        cv=cv,
        params={"groups": groups},
    )

    score = balanced_accuracy_score(y, pred)
    chance = 1.0 / len(modalities)
    cm = pd.DataFrame(
        confusion_matrix(y, pred, labels=modalities),
        index=modalities,
        columns=modalities,
    )
    return score, chance, cm


def internal_classifier_score(trainer, z_dict):
    """Evaluate the final XModalix internal adversarial classifier on frozen z."""
    modality_order = list(trainer._modality_dynamics.keys())
    X_parts, y_parts = [], []

    for label, modality in enumerate(modality_order):
        z = np.asarray(z_dict[modality])
        X_parts.append(z)
        y_parts.append(np.full(len(z), label, dtype=np.int64))

    X = torch.as_tensor(
        np.concatenate(X_parts, axis=0),
        dtype=torch.float32,
        device=trainer._fabric.device,
    )
    y = np.concatenate(y_parts, axis=0)

    trainer._latent_clf.eval()
    with torch.inference_mode():
        pred = trainer._latent_clf(X).argmax(dim=1).detach().cpu().numpy()

    score = balanced_accuracy_score(y, pred)
    cm = pd.DataFrame(
        confusion_matrix(y, pred, labels=np.arange(len(modality_order))),
        index=modality_order,
        columns=modality_order,
    )
    return score, 1.0 / len(modality_order), cm


def mean_posterior_sigma(logvar_dict):
    values = [np.exp(0.5 * np.asarray(logvar)).mean() for logvar in logvar_dict.values()]
    return float(np.mean(values))


def get_sub_loss_value(key, epoch, split="train"):
    dynamics = xmodal3d.result.sub_losses.get(key)
    if dynamics is None:
        return np.nan
    value = dynamics.get(epoch=epoch, split=split)
    if value is None:
        return np.nan
    return float(np.asarray(value).squeeze())


## Final frozen alignment summary

This is the main numerical output. `external_probe_*` should approach chance (1/3), whereas `alignment_ratio_*` should be below 1. `internal_classifier_z` is shown next to the external probe because a large discrepancy between the two indicates that the adversarial classifier is not exploiting modality information that remains present in the latent space.

Detailed confusion matrices are stored in `external_probe_confusions` and `internal_classifier_confusions`; detailed modality-pair results are stored in `final_pairwise_reports`.


In [ ]:
final_embeddings = {}
for split_name in ["train", "valid", "test"]:
    dataset = getattr(xmodal3d._datasets, split_name)
    final_embeddings[split_name] = extract_frozen_embeddings(
        trainer=xmodal3d._trainer,
        dataset=dataset,
        batch_size=config.batch_size,
    )

external_probe_confusions = {}
internal_classifier_confusions = {}
final_pairwise_reports = {}
summary_rows = []

for split_name, (mu, z, logvar, ids) in final_embeddings.items():
    mu_probe, chance, mu_cm = modality_probe_score(mu, ids)
    z_probe, _, z_cm = modality_probe_score(z, ids)
    internal_score, _, internal_cm = internal_classifier_score(xmodal3d._trainer, z)

    mu_pairwise = pairwise_alignment_report_exact(mu, ids)
    z_pairwise = pairwise_alignment_report_exact(z, ids)

    external_probe_confusions[(split_name, "mu")] = mu_cm
    external_probe_confusions[(split_name, "z")] = z_cm
    internal_classifier_confusions[split_name] = internal_cm
    final_pairwise_reports[(split_name, "mu")] = mu_pairwise
    final_pairwise_reports[(split_name, "z")] = z_pairwise

    summary_rows.append({
        "split": split_name,
        "external_probe_mu": mu_probe,
        "external_probe_z": z_probe,
        "internal_classifier_z": internal_score,
        "chance": chance,
        "alignment_ratio_mu": mu_pairwise["matched/unmatched_ratio"].mean(),
        "alignment_ratio_z": z_pairwise["matched/unmatched_ratio"].mean(),
        "mean_posterior_sigma": mean_posterior_sigma(logvar),
    })

final_alignment_summary = pd.DataFrame(summary_rows).set_index("split")
display(final_alignment_summary.round(4))
print("Use VALID for model decisions; TEST is retained for final reporting, not tuning.")


## Validation trajectories and training-health summary

Only one compact figure is produced:

1. external modality separability on validation checkpoints,
2. mean patient-specific alignment ratio on validation checkpoints,
3. the three XModalix alignment-related loss components,
4. mean posterior sigma on validation checkpoints.

The final reconstruction/variational losses and effective beta are shown as one compact row rather than separate plots.


In [ ]:
# Validation checkpoint epochs: the model is fixed during validation,
# so these checkpoint embeddings are suitable for trajectory diagnostics.
available_epochs = sorted(
    epoch
    for epoch, split_data in xmodal3d.result.mus._data.items()
    if split_data is not None
    and split_data.get("valid") is not None
)
assert available_epochs, "No stored validation checkpoints were found."

trajectory_rows = []
for epoch in available_epochs:
    valid_ids = xmodal3d.result.sample_ids.get(epoch=epoch, split="valid")
    valid_mu = xmodal3d.result.mus.get(epoch=epoch, split="valid")
    valid_z = xmodal3d.result.latentspaces.get(epoch=epoch, split="valid")
    valid_logvar = xmodal3d.result.sigmas.get(epoch=epoch, split="valid")

    mu_probe, chance, _ = modality_probe_score(valid_mu, valid_ids)
    z_probe, _, _ = modality_probe_score(valid_z, valid_ids)
    mu_pair = pairwise_alignment_report_exact(valid_mu, valid_ids)
    z_pair = pairwise_alignment_report_exact(valid_z, valid_ids)

    trajectory_rows.append({
        "epoch": epoch + 1,
        "probe_mu": mu_probe,
        "probe_z": z_probe,
        "chance": chance,
        "alignment_ratio_mu": mu_pair["matched/unmatched_ratio"].mean(),
        "alignment_ratio_z": z_pair["matched/unmatched_ratio"].mean(),
        "mean_sigma": mean_posterior_sigma(valid_logvar),
    })

trajectory_df = pd.DataFrame(trajectory_rows)

# Loss trajectories used only for the compact plot and final loss row.
modalities = ["img.NCCT", "img.CBF", "img.CBV"]
loss_rows = []
for epoch in range(config.epochs):
    recon_values, var_values, beta_values = [], [], []
    for modality in modalities:
        recon_values.append(get_sub_loss_value(f"{modality}.recon_loss", epoch))
        var_values.append(get_sub_loss_value(f"{modality}.var_loss", epoch))
        beta_values.append(get_sub_loss_value(f"{modality}.effective_beta_factor", epoch))

    loss_rows.append({
        "epoch": epoch + 1,
        "adver_loss": get_sub_loss_value("adver_loss", epoch),
        "paired_loss": get_sub_loss_value("paired_loss", epoch),
        "clf_loss": get_sub_loss_value("clf_loss", epoch),
        "mean_recon_loss": np.nanmean(recon_values),
        "mean_var_loss": np.nanmean(var_values),
        "mean_effective_beta": np.nanmean(beta_values),
    })

loss_over_time = pd.DataFrame(loss_rows)
final_loss_summary = loss_over_time.tail(1).set_index("epoch")
display(final_loss_summary.round(6))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1) Modality separability
ax = axes[0, 0]
ax.plot(trajectory_df["epoch"], trajectory_df["probe_mu"], marker="o", label="External probe: mu")
ax.plot(trajectory_df["epoch"], trajectory_df["probe_z"], marker="o", label="External probe: z")
ax.axhline(trajectory_df["chance"].iloc[0], linestyle="--", label="Chance")
valid_internal = final_alignment_summary.loc["valid", "internal_classifier_z"]
ax.scatter([trajectory_df["epoch"].iloc[-1]], [valid_internal], marker="x", s=80, label="Internal classifier: z (final)")
ax.set_title("Validation modality separability")
ax.set_xlabel("Joint-training epoch")
ax.set_ylabel("Balanced accuracy")
ax.legend()

# 2) Patient-specific cross-modal alignment
ax = axes[0, 1]
ax.plot(trajectory_df["epoch"], trajectory_df["alignment_ratio_mu"], marker="o", label="mu")
ax.plot(trajectory_df["epoch"], trajectory_df["alignment_ratio_z"], marker="o", label="z")
ax.axhline(1.0, linestyle="--", label="No patient-specific alignment")
ax.set_title("Validation patient-specific alignment")
ax.set_xlabel("Joint-training epoch")
ax.set_ylabel("Matched / unmatched distance")
ax.legend()

# 3) Alignment-related training losses
ax = axes[1, 0]
for column in ["adver_loss", "paired_loss", "clf_loss"]:
    ax.plot(loss_over_time["epoch"], loss_over_time[column], label=column)
ax.set_title("XModalix alignment-related losses")
ax.set_xlabel("Joint-training epoch")
ax.set_ylabel("Loss")
ax.legend()

# 4) Posterior uncertainty
ax = axes[1, 1]
ax.plot(trajectory_df["epoch"], trajectory_df["mean_sigma"], marker="o")
ax.set_title("Validation posterior uncertainty")
ax.set_xlabel("Joint-training epoch")
ax.set_ylabel("Mean posterior sigma")

plt.tight_layout()
plt.show()


## Optional detailed inspection

The notebook intentionally does not print confusion matrices, all modality-pair tables, full checkpoint tables, separate `mu`/`z` figures, beta-annealing figures, log-variance figures or raw loss tables. Those data remain available in `external_probe_confusions`, `internal_classifier_confusions`, `final_pairwise_reports`, `trajectory_df` and `loss_over_time` if a specific run requires deeper inspection.
